# Customer Segmentation Lab -- Module 2, Class 6

**Goal:** Segment customers by purchasing behavior using K-Means clustering.

This is the closest thing to what a data analyst or ML engineer actually does on the job.

Pipeline:
1. Build customer-level features
2. Scale features with StandardScaler
3. Find optimal k using the Elbow Method
4. Fit K-Means and assign clusters
5. Visualize segments
6. **TODO:** Interpret segments and write business recommendations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

---
## Step 1: Load and Build Customer Features

In [ ]:
# Load dataset
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Upload your CSV manually.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

print(f"Columns: {list(df.columns)}")

In [ ]:
# Identify key column names
# Adjust these if your dataset has different column names
customer_col = [c for c in df.columns if 'customer' in c.lower() and 'id' in c.lower()]
sales_col = [c for c in df.columns if c.lower() == 'sales']
profit_col = [c for c in df.columns if c.lower() == 'profit']
discount_col = [c for c in df.columns if c.lower() == 'discount']
quantity_col = [c for c in df.columns if c.lower() == 'quantity']

cust_id = customer_col[0] if customer_col else None
sales = sales_col[0] if sales_col else None
profit = profit_col[0] if profit_col else None
discount = discount_col[0] if discount_col else None
quantity = quantity_col[0] if quantity_col else None

print(f"Customer ID: {cust_id}")
print(f"Sales: {sales}, Profit: {profit}, Discount: {discount}, Quantity: {quantity}")

In [ ]:
# Build customer-level summary
customer_df = df.groupby(cust_id).agg(
    total_spending=(sales, 'sum'),
    order_count=(sales, 'count'),
    avg_order_value=(sales, 'mean'),
    avg_discount=(discount, 'mean'),
    total_profit=(profit, 'sum')
).reset_index()

print(f"Customer summary: {customer_df.shape[0]} customers")
customer_df.head()

In [ ]:
customer_df.describe()

---
## Step 2: Feature Selection and Scaling

We select features that capture purchasing behavior, then scale them so no single feature dominates the distance calculations.

**Why scaling matters:** total_spending might range from 100 to 50,000, while avg_discount ranges from 0 to 0.8. Without scaling, spending would completely dominate the clustering.

In [ ]:
# Select features for clustering
feature_names = ['total_spending', 'order_count', 'avg_order_value', 'avg_discount']
features = customer_df[feature_names]

# Scale with StandardScaler (mean=0, std=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("Before scaling:")
print(features.describe().round(2))
print("\nAfter scaling (first 5 rows):")
print(pd.DataFrame(features_scaled, columns=feature_names).head())

---
## Step 3: Elbow Method -- Find Optimal k

Run K-Means for k=2 through k=8. Plot the inertia (within-cluster sum of squares). Look for the "elbow" where adding more clusters stops helping much.

In [ ]:
inertias = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(features_scaled)
    inertias.append(km.inertia_)
    print(f"k={k}: inertia={km.inertia_:,.0f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), inertias, marker='o', linewidth=2, color='steelblue')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
ax.set_title('Elbow Method for Optimal k')
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

print("Pick the k where the curve bends -- adding more clusters gives diminishing returns.")

---
## Step 4: Fit Final K-Means Model

Based on the elbow plot, we use k=3 (adjust if the elbow suggests otherwise).

In [ ]:
# Fit with chosen k
k = 3  # Adjust based on your elbow plot
km_final = KMeans(n_clusters=k, random_state=42, n_init=10)
customer_df['segment'] = km_final.fit_predict(features_scaled)

print(f"Customers per segment:")
print(customer_df['segment'].value_counts().sort_index())

---
## Step 5: Visualization

In [ ]:
# Scatter plot: total_spending vs order_count, colored by segment
fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
for seg in sorted(customer_df['segment'].unique()):
    mask = customer_df['segment'] == seg
    ax.scatter(
        customer_df.loc[mask, 'total_spending'],
        customer_df.loc[mask, 'order_count'],
        label=f'Segment {seg}',
        alpha=0.6,
        color=colors[seg % len(colors)],
        s=50
    )

ax.set_xlabel('Total Spending ($)')
ax.set_ylabel('Order Count')
ax.set_title('Customer Segments: Spending vs Order Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: average feature values per segment
segment_means = customer_df.groupby('segment')[feature_names].mean()

fig, axes = plt.subplots(1, len(feature_names), figsize=(16, 5))

for i, feat in enumerate(feature_names):
    axes[i].bar(
        segment_means.index.astype(str),
        segment_means[feat],
        color=[colors[j % len(colors)] for j in segment_means.index],
        edgecolor='black'
    )
    axes[i].set_title(feat)
    axes[i].set_xlabel('Segment')

plt.suptitle('Average Feature Values per Segment', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster centers (in original scale for interpretation)
centers_scaled = km_final.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=feature_names)
centers_df.index.name = 'segment'

print("Cluster Centers (original scale):")
print(centers_df.round(2))

---
## TODO: Step 6 -- Interpret the Segments

Look at the cluster centers table and the visualizations above.

For EACH segment, write:
1. A descriptive name (e.g., "Budget Occasional Buyers", "High-Value Loyalists")
2. A 2-3 sentence description of the segment's behavior based on the numbers
3. How big this segment is (what % of total customers)

### Segment 0: [TODO: Name]

*TODO: Describe this segment. What are their spending patterns? How often do they order? Do they use discounts?*

### Segment 1: [TODO: Name]

*TODO: Describe this segment.*

### Segment 2: [TODO: Name]

*TODO: Describe this segment.*

---
## TODO: Step 7 -- Business Recommendations

For each segment, suggest ONE specific marketing action.

Be concrete. "Do more marketing" is not actionable.

Think about the Uzbekistan context: Telegram is the dominant messaging platform. SMS is universal. Email is less common for consumer outreach.

| Segment | Name | Recommended Action |
|---------|------|-------------------|
| 0 | [TODO] | [TODO: specific action, channel, offer] |
| 1 | [TODO] | [TODO: specific action, channel, offer] |
| 2 | [TODO] | [TODO: specific action, channel, offer] |

---
## TODO: Step 8 -- Limitations

List at least 3 limitations of this segmentation approach. Think about:
- Data quality and completeness
- Feature selection (what is missing?)
- Algorithm assumptions (K-Means assumes spherical clusters)
- What the model cannot capture (recency, product preferences, returns, complaints)

Write your limitations below:

1. *TODO: Limitation 1*
2. *TODO: Limitation 2*
3. *TODO: Limitation 3*

---
## Reflection Questions

Answer these below:

1. If you ran K-Means 10 times without setting `random_state`, would you get the same clusters every time? Why or why not?
2. You used K-Means, which assumes spherical clusters. What if your customer segments are not spherical? What alternative algorithm could you try?
3. A marketing manager says "I want 7 segments instead." How would you respond? What are the tradeoffs?

*TODO: Write your answers here.*